<a href="https://colab.research.google.com/github/YugamSachdeva/FarmDirect-AndroidApp/blob/main/ml_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install pandas scikit-learn joblib

In [23]:
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [24]:
import pandas as pd
import random

products = ["Wheat", "Rice", "Tomato", "Potato", "Onion", "Carrot", "Corn"]
categories = ["Grain", "Vegetable"]
seasons = ["Winter", "Summer", "Monsoon", "All"]
locations = ["Delhi", "Punjab", "UP", "Bihar", "Maharashtra", "Haryana"]

data = []

for _ in range(1000):  # 👉 change to 5000 if you want bigger
    product = random.choice(products)

    if product in ["Wheat", "Rice", "Corn"]:
        category = "Grain"
    else:
        category = "Vegetable"

    season = random.choice(seasons)
    location = random.choice(locations)

    quantity = random.randint(10, 500)
    is_organic = random.choice([0, 1])
    demand_index = round(random.uniform(0.5, 1.0), 2)

    # 💡 Smart price logic (IMPORTANT)
    base_price = {
        "Wheat": 20,
        "Rice": 30,
        "Tomato": 25,
        "Potato": 18,
        "Onion": 22,
        "Carrot": 24,
        "Corn": 21
    }[product]

    price = base_price

    # demand increases price
    price += demand_index * 10

    # organic increases price
    if is_organic:
        price += 5

    # quantity discount
    if quantity > 200:
        price -= 3

    # season effect
    if season == "Monsoon":
        price += 2

    price = round(price, 2)

    data.append([
        product,
        category,
        season,
        location,
        quantity,
        is_organic,
        demand_index,
        price
    ])

columns = [
    "product_name",
    "category",
    "season",
    "location",
    "quantity_kg",
    "is_organic",
    "demand_index",
    "target_price"
]

df = pd.DataFrame(data, columns=columns)

df.to_csv("sample_price_data.csv", index=False)

df.head()

,product_name,category,season,location,quantity_kg,is_organic,demand_index,target_price
0,Wheat,Grain,Summer,Bihar,445,1,0.78,29.8
1,Onion,Vegetable,Winter,Delhi,407,0,0.65,25.5
2,Carrot,Vegetable,All,Bihar,194,1,0.98,38.8
3,Corn,Grain,Winter,Haryana,81,0,0.63,27.3
4,Carrot,Vegetable,Winter,UP,136,1,0.97,38.7


In [25]:
df.shape

(1000, 8)

In [26]:
FEATURE_COLUMNS = [
    "product_name",
    "category",
    "season",
    "location",
    "quantity_kg",
    "is_organic",
    "demand_index",
]

TARGET_COLUMN = "target_price"


def build_pipeline():
    categorical_features = ["product_name", "category", "season", "location"]
    numeric_features = ["quantity_kg", "is_organic", "demand_index"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", "passthrough", numeric_features),
        ]
    )

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_split=4,
        random_state=42,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

In [27]:
df = pd.read_csv("sample_price_data.csv")

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = build_pipeline()
pipeline.fit(X_train, y_train)

predictions = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5
r2 = r2_score(y_test, predictions)

joblib.dump(pipeline, "price_model.joblib")

print("Training complete")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.2f}")

Training complete
MAE: 0.49
RMSE: 0.67
R2 Score: 0.98


In [28]:
def predict_price(
    product_name,
    category,
    season,
    location,
    quantity_kg,
    is_organic,
    demand_index
):
    model = joblib.load("price_model.joblib")

    sample = pd.DataFrame([{
        "product_name": product_name,
        "category": category,
        "season": season,
        "location": location,
        "quantity_kg": quantity_kg,
        "is_organic": is_organic,
        "demand_index": demand_index,
    }])

    prediction = model.predict(sample)[0]
    return prediction

In [29]:
price = predict_price(
    product_name="Tomato",
    category="Vegetable",
    season="Summer",
    location="UP",
    quantity_kg=60,
    is_organic=0,
    demand_index=0.85
)

print(f"Predicted Price: Rs {price:.2f} per kg")

Predicted Price: Rs 33.65 per kg
